In [4]:
from transformers import T5EncoderModel, T5Tokenizer
import torch
import numpy as np

In [8]:
tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-xxl")

In [9]:
model = T5EncoderModel.from_pretrained("google/t5-v1_1-xxl", device_map="auto", dtype=torch.bfloat16)

Loading weights: 100%|██████████| 220/220 [00:05<00:00, 37.71it/s]
[transformers] T5EncoderModel LOAD REPORT from: google/t5-v1_1-xxl
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
instruction = ["The rain is heavy today", "The sun is shining brightly.", "The car is parked in the garage."]

In [11]:
model.eval()
model.requires_grad_(False)

T5EncoderModel(
  (shared): Embedding(32128, 4096)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 4096)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=4096, out_features=4096, bias=False)
              (k): Linear(in_features=4096, out_features=4096, bias=False)
              (v): Linear(in_features=4096, out_features=4096, bias=False)
              (o): Linear(in_features=4096, out_features=4096, bias=False)
              (relative_attention_bias): Embedding(32, 64)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseGatedActDense(
              (wi_0): Linear(in_features=4096, out_features=10240, bias=False)
              (wi_1): Linear(in_features=4096, out_features=10240, bias=False)
              (wo

In [12]:
tokenized = tokenizer(instruction, padding=True, truncation=True, return_tensors="pt")

In [13]:
input_ids = tokenized["input_ids"].to(model.device)
attention_mask = tokenized["attention_mask"].to(model.device)

In [14]:
with torch.no_grad():
    text_embeddings = model(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state

In [15]:
text_embeddings.shape

torch.Size([3, 10, 4096])

In [16]:
import torch.nn.functional as F

def mean_pool(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1e-9)
    return summed / counts

sentence_embeddings = mean_pool(text_embeddings, attention_mask)

# Normalize so matrix multiplication produces cosine similarity
sentence_embeddings = F.normalize(
    sentence_embeddings.float(),
    p=2,
    dim=1,
)

similarity_matrix = sentence_embeddings @ sentence_embeddings.T
similarity_matrix.cpu().numpy()

array([[0.9999999 , 0.55672705, 0.40340045],
       [0.55672705, 0.99999994, 0.6555078 ],
       [0.40340045, 0.6555078 , 1.        ]], dtype=float32)

In [17]:
model.eval()

with torch.no_grad():
    output1 = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).last_hidden_state

    output2 = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).last_hidden_state

print(torch.equal(output1, output2))

True
